[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/autogen/autogen_agents_parallelization.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# AutoGen 智能体：并行执行

本教程使用 [AutoGen](https://microsoft.github.io/autogen/stable//index.html) 探索并行任务执行，并展示如何通过 Phoenix 追踪工作流。

并行执行让多个任务同时运行，显著缩短整体耗时，适用于任务独立且能够同时处理的场景。追踪可以帮助观察并发流程与各分支耗时。

AutoGen 没有内置并行执行管理器，但其核心智能体能力可以与标准 Python 并发库结合，同时启动多次智能体交互。

完成本教程后，你将学会：

- 设置执行特定任务的基础 AutoGen 智能体。

- 使用 Python threading 库并行执行一组智能体任务。

- 使用 Phoenix 追踪并行智能体交互。

- 比较并行与顺序智能体工作流。

⚠️ 本教程需要 OpenAI 密钥。

## 设置密钥与依赖

In [ ]:
!pip install -qq pyautogen==0.9 autogen-agentchat~=0.2

In [ ]:
!pip install -qqq arize-phoenix arize-phoenix-otel openinference-instrumentation-openai

In [ ]:
import os
from getpass import getpass

import autogen

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="autogen-agents",
    auto_instrument=True,
)

## 并行示例任务：产品描述生成器


## 定义并行智能体

本节定义由专业 `AssistantAgent` 处理多个任务的并行工作流。每个任务在独立线程运行，追踪捕获各任务的跨度及耗时。每个线程创建临时 `UserProxyAgent`，管理与智能体相互隔离的对话。

每当任务完成，就通过锁将生成内容与任务元数据收集到共享结果列表中，实现并发 LLM 交互。

本例调用营销智能体，生成智能手表产品描述的不同组成部分：功能、价值主张、目标客户与宣传语。

![流程图](https://storage.googleapis.com/arize-phoenix-assets/assets/images/arize_autogen_parallelization.png)

`llm_config` 指定所有助手智能体使用的配置。


In [ ]:
llm_config = {
    "config_list": [
        {
            "model": "gpt-4",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        }
    ]
}

In [ ]:
# 专业 LLM AssistantAgent
import threading

marketing_writer_agent = autogen.AssistantAgent(
    name="MarketingWriter",
    llm_config=llm_config,
    system_message="你是文案简洁、善于说服的营销撰稿人。仅根据收到的具体提示词生成内容，回复聚焦所需信息。",
)

results = []
results_lock = threading.Lock()

In [ ]:
import time

import opentelemetry.context as context_api
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

tracer = tracer_provider.get_tracer(__name__)


def run_agent_task_with_tracing(agent, task_prompt, task_id, parent_context):
    token = context_api.attach(parent_context)
    thread_name = threading.current_thread().name

    with tracer.start_as_current_span(task_id, openinference_span_kind="agent") as child_span:
        start_time = time.time()

        # 为当前线程创建临时 UserProxyAgent
        temp_user_proxy = autogen.UserProxyAgent(
            name=f"UserProxy_{thread_name}",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=1,
            is_termination_msg=lambda x: True,
            code_execution_config=False,
        )

        # 启动对话
        temp_user_proxy.initiate_chat(
            agent,
            message=task_prompt,
            clear_history=True,
            silent=True,
        )

        # 获取最后一条消息
        assistant_reply = "未找到回复。"
        if temp_user_proxy.last_message(agent):
            assistant_reply = temp_user_proxy.last_message(agent).get("content", "未找到回复。")
        child_span.set_status(Status(StatusCode.OK))

        end_time = time.time()
        duration = end_time - start_time
        print(f"线程 {task_id}: 完成耗时 {duration:.2f} 秒。")

        child_span.set_status(trace.StatusCode.OK)

        results.append(
            {"task_id": task_id, "generated_content": assistant_reply, "duration": duration}
        )

    context_api.detach(token)

### 运行并追踪智能体

接下来运行智能体。全部线程结束后，收集结果，按原始任务顺序排序并打印。

In [ ]:
start_total_time = time.time()

results = []

with tracer.start_as_current_span("Parallelized", openinference_span_kind="agent") as parent_span:
    tasks = [
        {"id": "Features", "prompt": "列出新款 SmartX Pro 智能手表的主要功能……"},
        {"id": "ValueProp", "prompt": "描述 SmartX Pro 的核心价值主张……"},
        {"id": "TargetCustomer", "prompt": "描述 SmartX Pro 的理想客户画像……"},
        {"id": "Tagline", "prompt": "为 SmartX Pro 创作 3 条吸引人的营销宣传语。"},
    ]

    parent_context_to_pass = context_api.get_current()

    threads = []
    for i, task_info in enumerate(tasks):
        thread = threading.Thread(
            target=run_agent_task_with_tracing,
            args=(
                marketing_writer_agent,
                task_info["prompt"],
                task_info["id"],
                parent_context_to_pass,
            ),
            name=f"{task_info['id']}",
        )
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

end_total_time = time.time()
total_duration = end_total_time - start_total_time
print(f"\n所有线程完成，总耗时 {total_duration:.2f} 秒。")

# --- 打印结果 ---
results.sort(key=lambda x: [t["id"] for t in tasks].index(x["task_id"]))
print("--- 结果 ---")
for result in results:
    print(f"--- {result['task_id']} ---")
    print(f"生成内容：\n{result['generated_content']}")

## 定义顺序智能体（延迟对比）

本节按顺序执行同一组内容生成任务，各任务依次运行，可直接比较顺序与并行工作流的总执行时间。

In [ ]:
results = []


def run_single_task(agent, task_prompt, task_id):
    with tracer.start_as_current_span(f"{task_id}", openinference_span_kind="agent") as span:
        start_time = time.time()

        # 创建临时 UserProxyAgent
        temp_user_proxy = autogen.UserProxyAgent(
            name="UserProxy_Sequential",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=1,
            is_termination_msg=lambda x: True,
            code_execution_config=False,
        )

        reply = "未找到回复。"
        temp_user_proxy.initiate_chat(
            agent,
            message=task_prompt,
            clear_history=True,
            silent=True,
        )

        # 获取最后一条消息
        if temp_user_proxy.last_message(agent):
            reply = temp_user_proxy.last_message(agent).get("content", "未找到回复。")
        span.set_status(Status(StatusCode.OK))

        end_time = time.time()
        duration = end_time - start_time
        print(f"已完成任务 '{task_id}' in {duration:.2f} 秒。")

        results.append({"task_id": task_id, "prompt": task_prompt, "generated_content": reply})

### 运行并追踪顺序智能体

In [ ]:
start_total_time = time.time()

with tracer.start_as_current_span("Sequential", openinference_span_kind="agent") as parent_span:
    # 定义任务
    tasks = [
        {
            "id": "Features",
            "prompt": "列出新款 SmartX Pro 智能手表的主要功能。请具体说明屏幕类型、续航、传感器和连接能力等。",
        },
        {
            "id": "ValueProp",
            "prompt": "描述 SmartX Pro 对忙碌专业人士的核心价值，重点说明它如何解决问题或改善生活。",
        },
        {
            "id": "TargetCustomer",
            "prompt": "描述 SmartX Pro 智能手表的理想客户画像，包括人口特征、生活方式与需求。",
        },
        {
            "id": "Tagline",
            "prompt": "为 SmartX Pro 创作 3 条吸引人且各不相同的营销宣传语。",
        },
    ]

    for i, task_info in enumerate(tasks):
        parent_span.add_event(f"开始任务 {i + 1}: {task_info['id']}")
        run_single_task(
            agent=marketing_writer_agent, task_prompt=task_info["prompt"], task_id=task_info["id"]
        )
        parent_span.add_event(f"完成任务 {i + 1}: {task_info['id']}")


end_total_time = time.time()
total_duration = end_total_time - start_total_time
print(f"\n所有任务已顺序完成，总耗时 {total_duration:.2f} 秒。")


print("--- 结果 ---")
for result in results:
    print(f"--- {result['task_id']} ---")
    print(f"生成内容：\n{result['generated_content']}")

## 在 Phoenix 中查看结果

下面的追踪结果着重展示并行工作流的性能。顺序版本的耗时为并行版本的 3–4 倍。在 Phoenix 中可以查看 LLM 输入、输出的详细分解，以及延迟、令牌数量、模型配置等元数据。

![结果](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/phoenix_autogen_parallelized.gif)